# Part 14: Natural Language Processing (NLP)

**Quick Reference for Text Processing & NLP**

[Back to Index](Index.ipynb)

---
## 14.1 Text Preprocessing

**Purpose:** Clean and prepare text data for modeling

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

text = "Natural Language Processing is amazing! It helps us understand text. Visit www.example.com for more."

# 1. Lowercasing
text_lower = text.lower()
print(f"Lowercase: {text_lower}")

# 2. Remove URLs
text_no_url = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

# 3. Remove special characters and numbers
text_clean = re.sub(r'[^a-zA-Z\s]', '', text_lower)
print(f"Clean text: {text_clean}")

# 4. Tokenization (split into words)
tokens = word_tokenize(text_clean)
print(f"Tokens: {tokens}")

# Sentence tokenization
sentences = sent_tokenize(text)
print(f"Sentences: {sentences}")

# 5. Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens if w not in stop_words]
print(f"After stopword removal: {filtered_tokens}")

# 6. Stemming (reduce to root form, faster but crude)
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in filtered_tokens]
print(f"Stemmed: {stemmed}")

# 7. Lemmatization (reduce to dictionary form, slower but accurate)
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w, pos='v') for w in filtered_tokens]
print(f"Lemmatized: {lemmatized}")

### Complete Preprocessing Pipeline

In [ ]:
def preprocess_text(text, stem=False, lemma=True):
    """Complete text preprocessing pipeline"""
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    
    # Stemming or Lemmatization
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(w) for w in tokens]
    elif lemma:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(w) for w in tokens]
    
    return ' '.join(tokens)

# Apply to DataFrame
import pandas as pd
df['clean_text'] = df['text'].apply(preprocess_text)

---
## 14.2 Bag of Words (BOW)

**Concept:** Represent text as vector of word counts

**Pros:** Simple, interpretable

**Cons:** Ignores word order, sparse vectors

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    'Machine learning is great',
    'Deep learning is powerful',
    'Machine learning and deep learning'
]

# Create BOW
vectorizer = CountVectorizer(
    max_features=1000,      # limit vocabulary size
    ngram_range=(1, 2),     # unigrams and bigrams
    min_df=1,               # minimum document frequency
    max_df=0.9              # maximum document frequency
)
X = vectorizer.fit_transform(documents)

print(f"Vocabulary: {vectorizer.get_feature_names_out()}")
print(f"BOW matrix shape: {X.shape}")
print(f"BOW matrix:\n{X.toarray()}")

# Convert to DataFrame for better visualization
bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(bow_df)

---
## 14.3 TF-IDF (Term Frequency-Inverse Document Frequency)

**Formula:** TF-IDF = TF(t,d) × IDF(t)
- TF = count(t in d) / total words in d
- IDF = log(total docs / docs containing t)

**Purpose:** Weight words by importance (downweight common words)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF
tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.9,
    sublinear_tf=True      # use log scaling for TF
)
X_tfidf = tfidf.fit_transform(documents)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"TF-IDF matrix:\n{X_tfidf.toarray()}")

# Get top TF-IDF terms for a document
feature_names = tfidf.get_feature_names_out()
doc_idx = 0
tfidf_scores = zip(feature_names, X_tfidf.toarray()[doc_idx])
top_terms = sorted(tfidf_scores, key=lambda x: x[1], reverse=True)[:5]
print(f"\nTop 5 terms for doc {doc_idx}: {top_terms}")

### Text Classification Example

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Assuming df has 'text' and 'label' columns
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

# Vectorize
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train classifier
clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

---
## 14.4 Word Embeddings

**Concept:** Dense vector representations of words capturing semantic meaning

**Advantage:** Similar words have similar vectors

### Word2Vec

In [ ]:
from gensim.models import Word2Vec
import numpy as np

# Prepare sentences (list of tokenized sentences)
sentences = [sent.split() for sent in df['clean_text']]

# Train Word2Vec
model = Word2Vec(
    sentences,
    vector_size=100,        # embedding dimension
    window=5,               # context window
    min_count=2,            # ignore words with freq < 2
    sg=1,                   # 1=skip-gram, 0=CBOW
    workers=4
)

# Get word vector
vector = model.wv['machine']
print(f"Vector for 'machine': {vector[:10]}...")  # first 10 dims

# Find similar words
similar = model.wv.most_similar('machine', topn=5)
print(f"Similar to 'machine': {similar}")

# Word arithmetic (king - man + woman ≈ queen)
result = model.wv.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
print(f"king - man + woman = {result}")

# Document embedding (average word vectors)
def document_vector(doc, model):
    vectors = [model.wv[word] for word in doc.split() if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

doc_vec = document_vector("machine learning is great", model)
print(f"Document vector shape: {doc_vec.shape}")

### Using Pre-trained Word2Vec (Google News)

In [ ]:
import gensim.downloader as api

# Load pre-trained model (first time will download ~1.5GB)
model = api.load('word2vec-google-news-300')

# Use directly
similar = model.most_similar('computer', topn=5)
print(f"Similar to 'computer': {similar}")

# Similarity
similarity = model.similarity('computer', 'laptop')
print(f"Similarity: {similarity:.3f}")

### GloVe (Global Vectors)

In [ ]:
# Load pre-trained GloVe
glove_model = api.load('glove-wiki-gigaword-100')

# Use same as Word2Vec
vector = glove_model['machine']
similar = glove_model.most_similar('machine', topn=5)
print(f"GloVe similar to 'machine': {similar}")

---
## 14.5 Advanced NLP Techniques

### Named Entity Recognition (NER)

In [ ]:
import spacy

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976."
doc = nlp(text)

# Extract named entities
for ent in doc.ents:
    print(f"{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}")

# Visualize
from spacy import displacy
displacy.render(doc, style='ent', jupyter=True)

### Part-of-Speech (POS) Tagging

In [ ]:
text = "Natural language processing is amazing"
doc = nlp(text)

for token in doc:
    print(f"{token.text:15} {token.pos_:10} {token.tag_:10} {spacy.explain(token.tag_)}")

### Sentiment Analysis

In [ ]:
from textblob import TextBlob

text = "This product is absolutely amazing! I love it."
blob = TextBlob(text)

# Polarity: -1 (negative) to 1 (positive)
# Subjectivity: 0 (objective) to 1 (subjective)
print(f"Polarity: {blob.sentiment.polarity:.3f}")
print(f"Subjectivity: {blob.sentiment.subjectivity:.3f}")

# Using VADER (better for social media)
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()
scores = sia.polarity_scores(text)
print(f"VADER scores: {scores}")
# neg, neu, pos, compound (-1 to 1)

### Topic Modeling (LDA)

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# Vectorize documents
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df['clean_text'])

# Train LDA
lda = LatentDirichletAllocation(
    n_components=5,         # number of topics
    random_state=42
)
lda.fit(X)

# Print topics
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

# Get topic distribution for documents
doc_topics = lda.transform(X)
print(f"\nTopic distribution for first doc: {doc_topics[0]}")

---
## 14.6 Modern NLP with Transformers

**Concept:** Pre-trained models fine-tuned for specific tasks

**Popular Models:** BERT, GPT, T5, RoBERTa

### Using Hugging Face Transformers

In [ ]:
from transformers import pipeline

# 1. Sentiment Analysis
sentiment_analyzer = pipeline('sentiment-analysis')
result = sentiment_analyzer("I love this product!")
print(f"Sentiment: {result}")

# 2. Text Generation
generator = pipeline('text-generation', model='gpt2')
result = generator("Machine learning is", max_length=50, num_return_sequences=1)
print(f"Generated: {result}")

# 3. Question Answering
qa_model = pipeline('question-answering')
context = "Paris is the capital of France. It has a population of 2.1 million."
question = "What is the capital of France?"
answer = qa_model(question=question, context=context)
print(f"Answer: {answer['answer']}")

# 4. Named Entity Recognition
ner = pipeline('ner', grouped_entities=True)
result = ner("Apple Inc. was founded by Steve Jobs in California.")
print(f"NER: {result}")

# 5. Text Summarization
summarizer = pipeline('summarization')
text = """Long text here..."""
summary = summarizer(text, max_length=100, min_length=30)
print(f"Summary: {summary}")

# 6. Translation
translator = pipeline('translation_en_to_fr')
result = translator("Hello, how are you?")
print(f"Translation: {result}")

### BERT for Text Classification

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

# Load pre-trained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize
texts = ["I love this product", "This is terrible"]
encodings = tokenizer(texts, truncation=True, padding=True, return_tensors='pt')

# Predict
with torch.no_grad():
    outputs = model(**encodings)
    predictions = torch.argmax(outputs.logits, dim=-1)
    print(f"Predictions: {predictions}")

# Fine-tuning (simplified)
# 1. Prepare dataset
# 2. Define training arguments
# 3. Train using Trainer API
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

---
### Quick Reference

**Method Comparison:**

| Method | Dimension | Semantics | Use Case |
|--------|-----------|-----------|----------|
| BOW | High (vocab size) | No | Simple, baseline |
| TF-IDF | High | No | Document retrieval |
| Word2Vec | Low (50-300) | Yes | Word similarity |
| GloVe | Low (50-300) | Yes | Word similarity |
| BERT | High (768) | Yes | State-of-the-art |

**When to use:**
- Simple classification → TF-IDF + Naive Bayes/LogReg
- Semantic understanding → Word2Vec/GloVe
- State-of-the-art → BERT/GPT (transformers)
- Limited data → Pre-trained models
- Real-time → Lightweight models (TF-IDF)

**NLP Pipeline:**
1. Text preprocessing (clean, tokenize, remove stopwords)
2. Feature extraction (BOW/TF-IDF/Embeddings)
3. Model training (classification/NER/etc.)
4. Evaluation (accuracy, F1, BLEU for translation)
5. Deployment